# Gap 1: $\mathcal{N}_\lambda$ vs Borel on Factorial/Divergent Series

**Motivation.** All our QED/QCD tests targeted series that are still in the convergent regime.
The graded SCN attenuation operator $\mathcal{N}_\lambda(x) = \lambda^{-\sigma(x)} x$ is
designed for poorly-behaved series — its natural habitat is factorial growth $c_n \sim n!$.

**Test cases:**
1. **Euler/Borel series:** $S(g) = \sum_{n=0}^\infty (-1)^n n! \, g^n$ — Borel-summable to $\int_0^\infty e^{-t}/(1+gt)\,dt$
2. **Anharmonic oscillator:** $E(g) = \sum a_n g^n$ where $a_n \sim (-1)^{n+1} \Gamma(n+1/2) \cdot 3^n / \pi^{3/2}$
3. **Renormalon-like series:** $\sum n! \left(\frac{\beta_0}{2\pi}\right)^n g^{n+1}$ (IR renormalon)

For each: compare $\mathcal{N}_\lambda$ summation, standard truncation $\mathcal{N}_k$, and Borel summation against the known exact answer.

**Success criterion:** $\mathcal{N}_\lambda$ matches or beats Borel summation for at least one test case.

In [ ]:
import numpy as np
from scipy import integrate
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'axes.grid': True})

## Test 1: The Euler Series $S(g) = \sum (-1)^n n! \, g^n$

This is the canonical example of a divergent but Borel-summable series.

**Exact answer** (Borel sum): $S(g) = \int_0^\infty \frac{e^{-t}}{1 + g t}\, dt$

For small $g$, this is well-defined and finite despite the series diverging for all $g > 0$.

In [ ]:
# ── Exact answer via Borel integral ───────────────────────────
def euler_exact(g):
    """Borel sum of Σ (-1)^n n! g^n = ∫₀^∞ e^{-t}/(1+gt) dt."""
    result, _ = integrate.quad(lambda t: np.exp(-t) / (1 + g * t), 0, np.inf)
    return result

# ── Standard truncation N_k ──────────────────────────────────
def euler_truncated(g, k):
    """Partial sum Σ_{n=0}^{k-1} (-1)^n n! g^n."""
    return sum((-1)**n * np.math.factorial(n) * g**n for n in range(k))

# ── Attenuation N_λ ──────────────────────────────────────────
def euler_attenuated(g, lam, N_terms=50):
    """Σ_{n=0}^{N-1} (-1)^n n! g^n λ^{-n}."""
    return sum((-1)**n * np.math.factorial(n) * g**n * lam**(-n) for n in range(N_terms))

# ── Borel-Padé summation (standard resummation) ──────────────
def euler_borel_pade(g, order=10):
    """Borel transform + Padé approximant.
    
    Borel transform: B(t) = Σ (-1)^n t^n = 1/(1+t)
    For this series, Borel is exact — it recovers 1/(1+t) at all orders.
    So Borel-Padé also gives the exact answer.
    """
    # For the Euler series, B(t) = 1/(1+t) exactly.
    # ∫ e^{-t} B(gt) dt = ∫ e^{-t}/(1+gt) dt = exact answer
    return euler_exact(g)

# Test at several coupling values
print(f"{'g':>6}  {'Exact':>12}  {'N=5':>12}  {'N=10':>12}  {'N=20':>12}  {'N_λ(λ*)':>12}")
print("-" * 72)

for g in [0.05, 0.1, 0.2, 0.5, 1.0, 2.0]:
    exact = euler_exact(g)
    t5 = euler_truncated(g, 5)
    t10 = euler_truncated(g, 10)
    t20 = euler_truncated(g, 20)
    
    # Find optimal λ for N_λ
    best_lam = 1.0
    best_err = float('inf')
    for lam in np.linspace(1.01, 100, 2000):
        try:
            val = euler_attenuated(g, lam, N_terms=50)
            err = abs(val - exact)
            if err < best_err:
                best_err = err
                best_lam = lam
        except (OverflowError, ValueError):
            continue
    
    att_val = euler_attenuated(g, best_lam, N_terms=50)
    
    print(f"{g:>6.2f}  {exact:>12.8f}  {t5:>12.4f}  {t10:>12.0f}  {t20:>12.0e}  {att_val:>12.8f}")

In [ ]:
# ── Key insight: N_λ makes the series converge ────────────────
# The attenuated series is Σ (-1)^n n! (g/λ)^n
# This converges iff g/λ < lim_{n→∞} 1/n! → it NEVER converges
# because n! grows faster than λ^n for any fixed λ.
#
# Wait — that's the key question. Let's check explicitly.

g = 0.1
print(f"Euler series at g={g}:")
print(f"Exact = {euler_exact(g):.10f}")
print()

print(f"{'N_terms':>7}  {'λ=1':>15}  {'λ=2':>15}  {'λ=5':>15}  {'λ=10':>15}  {'λ=50':>15}")
print("-" * 82)

for N in [5, 10, 15, 20, 25, 30, 40, 50]:
    vals = []
    for lam in [1, 2, 5, 10, 50]:
        try:
            v = euler_attenuated(g, lam, N)
            if abs(v) > 1e15:
                vals.append(f"{'diverges':>15}")
            else:
                vals.append(f"{v:>15.8f}")
        except OverflowError:
            vals.append(f"{'overflow':>15}")
    print(f"{N:>7}  {'  '.join(vals)}")

print()
print("The n! growth defeats any fixed λ: the series still diverges.")
print("N_λ can only delay the divergence, not cure it.")
print("This is fundamentally different from Borel, which removes the n! entirely.")

In [ ]:
# ── Optimal truncation + attenuation ──────────────────────────
# Even if N_λ can't make the series converge, maybe it can improve
# the *optimal truncation* — i.e., truncate at the smallest term
# with attenuation helping to push that point further out.

def optimal_truncation(g, lam=1.0, max_n=100):
    """Find the truncation order where |term_n| is minimized."""
    terms = []
    partial_sum = 0.0
    for n in range(max_n):
        try:
            term = (-1)**n * np.math.factorial(n) * (g/lam)**n
            terms.append((n, term, partial_sum))
            partial_sum += term
        except (OverflowError, ValueError):
            break
    
    # Find smallest |term|
    if not terms:
        return None
    min_idx = min(range(len(terms)), key=lambda i: abs(terms[i][1]))
    n_opt, _, partial_at_opt = terms[min_idx]
    # Partial sum including up to n_opt
    partial_sum = sum(t[1] for t in terms[:n_opt+1])
    return n_opt, partial_sum, abs(terms[min_idx][1])

print("Optimal truncation: standard vs attenuated")
print(f"{'g':>5}  {'Exact':>12}  {'λ':>5}  {'n_opt':>5}  {'S(n_opt)':>12}  {'|error|':>12}  {'|min term|':>12}")
print("-" * 75)

for g in [0.05, 0.1, 0.2, 0.5, 1.0]:
    exact = euler_exact(g)
    for lam in [1.0, 2.0, 5.0, 10.0]:
        result = optimal_truncation(g, lam)
        if result:
            n_opt, s_opt, min_term = result
            # The partial sum uses g/λ, need to compare with exact at g
            # The attenuated partial sum IS the value at g/λ scaled
            # Actually: Σ_{n=0}^{N} (-1)^n n! g^n λ^{-n} = Σ (-1)^n n! (g/λ)^n
            # This is the Euler series evaluated at g_eff = g/λ, not at g!
            # So we should compare with euler_exact(g/lam)?? No.
            # N_λ is supposed to approximate the ORIGINAL function at g, not at g/λ.
            # But the series at g/λ converges toward euler_exact(g/λ), not euler_exact(g).
            # This is the degeneracy problem: loop-order N_λ = coupling rescaling.
            err = abs(s_opt - exact)
            print(f"{g:>5.2f}  {exact:>12.8f}  {lam:>5.1f}  {n_opt:>5}  {s_opt:>12.8f}  {err:>12.2e}  {min_term:>12.2e}")
    print()

In [ ]:
# ── Critical realization: N_λ on a factorially divergent series ──
#
# For Σ c_n g^n with c_n ~ n!, the attenuated series Σ c_n g^n λ^{-n}
# = Σ c_n (g/λ)^n still diverges (n! beats any geometric λ^n).
#
# Borel works because it removes the n!: B(t) = Σ c_n/n! t^n converges.
# N_λ doesn't touch the n! at all — it only rescales g.
#
# So the question is: can we define a DEPTH-DEPENDENT attenuation
# that grows fast enough to tame n! ?
#
# The framework gives us N_λ(x) = λ^{-σ(x)} x with σ additive.
# For loop order σ=n, this gives λ^{-n} which loses to n!.
#
# What if σ is NOT loop order but something that grows faster?
# E.g., if σ(diagram at loop n) counts the number of self-energy
# sub-insertions, and diagrams contributing to n! growth are precisely
# the deeply-nested ones with σ ~ n?
#
# Then we'd need λ^{-σ} where σ ~ n, giving λ^{-n} — same problem.
#
# Unless σ grows SUPER-linearly: σ ~ n log n or σ ~ n².
# But that breaks the additive propagation rule σ(xy) = σ(x) + σ(y).

# Let's test a generalized attenuation: weight_n = exp(-α * n * log(n))
# This is NOT in the framework (breaks multiplicativity) but tests
# whether ANY order-dependent weighting can beat Borel.

def euler_generalized_weight(g, alpha_w, N_terms=200):
    """Σ (-1)^n n! g^n exp(-α n ln n), i.e., weight = n^{-αn}."""
    total = 1.0  # n=0 term
    for n in range(1, N_terms):
        try:
            log_term = np.log(np.math.factorial(n)) + n * np.log(g) - alpha_w * n * np.log(n)
            if log_term > 300:  # overflow
                break
            term = (-1)**n * np.exp(log_term)
            total += term
        except (OverflowError, ValueError):
            break
    return total

g = 0.2
exact = euler_exact(g)
print(f"Euler series at g={g}, exact = {exact:.10f}")
print()
print(f"{'α_w':>6}  {'S_weighted':>15}  {'|error|':>12}  {'vs optimal trunc':>18}")
print("-" * 55)

opt_result = optimal_truncation(g, 1.0)
opt_err = abs(opt_result[1] - exact) if opt_result else float('inf')

for alpha_w in np.linspace(0.1, 3.0, 20):
    val = euler_generalized_weight(g, alpha_w)
    err = abs(val - exact)
    ratio = err / opt_err if opt_err > 0 else float('inf')
    print(f"{alpha_w:>6.2f}  {val:>15.10f}  {err:>12.2e}  {ratio:>15.4f}×")

In [ ]:
# ── Systematic comparison: N_λ vs Borel vs optimal truncation ──
# across a range of coupling g

gs = np.linspace(0.01, 1.0, 50)
err_optimal = []
err_borel = []
err_nla_best = []    # best N_λ (optimized λ, fixed truncation)
err_gen_best = []    # best generalized weight (optimized α_w)

for g in gs:
    exact = euler_exact(g)
    
    # Optimal truncation (standard)
    opt = optimal_truncation(g, 1.0)
    e_opt = abs(opt[1] - exact) if opt else 1e10
    err_optimal.append(e_opt)
    
    # Borel (exact for this series)
    err_borel.append(0.0)  # Borel is exact here
    
    # Best N_λ (optimize λ, use optimal truncation at g/λ)
    best_nla = 1e10
    for lam in np.linspace(1.0, 50, 200):
        r = optimal_truncation(g, lam)
        if r:
            e = abs(r[1] - exact)
            if e < best_nla:
                best_nla = e
    err_nla_best.append(best_nla)
    
    # Best generalized weight
    best_gen = 1e10
    for alpha_w in np.linspace(0.5, 3.0, 50):
        val = euler_generalized_weight(g, alpha_w)
        e = abs(val - exact)
        if e < best_gen:
            best_gen = e
    err_gen_best.append(best_gen)

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(gs, err_optimal, 'b-', label='Optimal truncation (standard)', linewidth=2)
ax.semilogy(gs, err_nla_best, 'r--', label=r'$\mathcal{N}_\lambda$ (best $\lambda$)', linewidth=2)
ax.semilogy(gs, err_gen_best, 'g-.', label=r'Generalized weight $n^{-\alpha n}$ (best $\alpha$)', linewidth=2)
ax.axhline(0, color='k', linestyle=':', alpha=0.3, label='Borel (exact)')
ax.set_xlabel('Coupling g')
ax.set_ylabel('|Error| vs exact answer')
ax.set_title('Euler series: summation methods comparison')
ax.legend()
ax.set_ylim(1e-16, 1e2)
plt.tight_layout()
plt.show()

## Test 2: Anharmonic Oscillator

Ground state energy $E_0(g) = \frac{1}{2} + \sum_{n=1}^\infty a_n g^n$ for $H = p^2/2 + x^2/2 + g x^4$.

Known large-order behavior: $a_n \sim (-1)^{n+1} \frac{\sqrt{6}}{\pi^{3/2}} 3^n \Gamma(n + 1/2)$

This is a Borel-summable alternating factorial series — harder than Euler because the
coefficients have non-trivial structure at low orders.

In [ ]:
# Known perturbative coefficients for the anharmonic oscillator
# H = p²/2 + x²/2 + g x⁴  (ground state energy)
# 
# a_n from Bender & Wu (1969, 1973), available to high order.
# Using the convention E = Σ a_n g^n with a_0 = 1/2.

# First ~20 coefficients (exact rational, from Bender-Wu recursion)
AHO_COEFFS = [
    1/2,       # a_0
    3/4,       # a_1
    -21/8,     # a_2
    333/16,    # a_3
    -30885/128,  # a_4
    916731/256,  # a_5
    -65518401/1024,  # a_6
    2723294673/2048,  # a_7
    -1030495099053/32768,  # a_8  
    54626982511455/65536,  # a_9  
    -6417007431590595/262144,  # a_10
]

print("Anharmonic oscillator coefficients a_n:")
for n, a in enumerate(AHO_COEFFS):
    print(f"  a_{n} = {a:>20.4f}    |a_n|/n! = {abs(a)/max(np.math.factorial(n),1):>12.4f}")

# Large-order asymptotic: |a_n| ~ sqrt(6)/π^(3/2) * 3^n * Γ(n+1/2)
print("\nLarge-order check (ratio |a_n| / [√6/π^{3/2} 3^n Γ(n+1/2)]):")
for n in range(2, len(AHO_COEFFS)):
    asymp = np.sqrt(6) / np.pi**1.5 * 3**n * np.exp(np.lgamma(n + 0.5))
    ratio = abs(AHO_COEFFS[n]) / asymp
    print(f"  n={n}: ratio = {ratio:.4f}")

In [ ]:
# ── Exact ground state energy via numerical diagonalization ───
from scipy.linalg import eigh_tridiagonal

def aho_exact(g, n_basis=200):
    """Exact E_0(g) via harmonic oscillator basis diagonalization."""
    # H = p²/2 + x²/2 + g x⁴ in HO basis
    # <n|x²|m> and <n|x⁴|m> matrix elements are known analytically
    n = np.arange(n_basis)
    
    # HO part: E_n = n + 1/2
    diag = n + 0.5
    
    # x⁴ matrix elements in HO basis:
    # <n|x⁴|n> = (3/4)(2n² + 2n + 1)
    # <n|x⁴|n+2> = (1/2)(n+1)(n+2) √... etc.
    # Using x = (a + a†)/√2, x⁴ gives Δn = 0, ±2, ±4
    
    # Full x⁴ matrix (dense, but n_basis is moderate)
    H = np.zeros((n_basis, n_basis))
    for i in range(n_basis):
        # diagonal
        H[i, i] = (i + 0.5) + g * 0.75 * (2*i*i + 2*i + 1)
        # off-diagonal Δn = 2
        if i + 2 < n_basis:
            val = g * 0.5 * np.sqrt((i+1)*(i+2)) * (2*i + 3)
            H[i, i+2] = val
            H[i+2, i] = val
        # off-diagonal Δn = 4
        if i + 4 < n_basis:
            val = g * 0.25 * np.sqrt((i+1)*(i+2)*(i+3)*(i+4))
            H[i, i+4] = val
            H[i+4, i] = val
    
    eigenvalues = np.linalg.eigvalsh(H)
    return eigenvalues[0]

# Verify at small g against perturbation theory
print("Verification: exact vs perturbative at small g")
for g in [0.001, 0.01, 0.05, 0.1, 0.5, 1.0]:
    exact = aho_exact(g)
    pert = sum(AHO_COEFFS[n] * g**n for n in range(len(AHO_COEFFS)))
    print(f"  g={g:<6.3f}  exact={exact:.8f}  pert(10)={pert:.8f}  diff={exact-pert:.2e}")

In [ ]:
# ── Borel summation for the anharmonic oscillator ────────────
def aho_borel_pade(g, coeffs, pade_order=None):
    """Borel-Padé summation of Σ a_n g^n.
    
    Borel transform: B(t) = Σ a_n/n! t^n
    Then S(g) = ∫_0^∞ e^{-t} B(gt) dt
    
    We use Padé approximant [N/N] or [(N+1)/N] for B(t).
    """
    from numpy.polynomial import polynomial as P
    
    # Borel transform coefficients
    borel_coeffs = [coeffs[n] / np.math.factorial(n) for n in range(len(coeffs))]
    
    if pade_order is None:
        pade_order = len(borel_coeffs) // 2
    
    # Padé [M/N] approximant
    M = pade_order
    N = min(pade_order, len(borel_coeffs) - pade_order - 1)
    if N < 1:
        N = 1
        M = len(borel_coeffs) - N - 1
    
    # Build Padé from Taylor coefficients using scipy
    from scipy.interpolate import pade as scipy_pade
    try:
        p_coeffs, q_coeffs = scipy_pade(borel_coeffs, N)
        
        def pade_func(t):
            num = np.polyval(p_coeffs[::-1], t)
            den = np.polyval(q_coeffs[::-1], t)
            if abs(den) < 1e-15:
                return np.nan
            return num / den
        
        # Integrate
        result, _ = integrate.quad(
            lambda t: np.exp(-t) * pade_func(g * t),
            0, np.inf, limit=200
        )
        return result
    except Exception:
        return np.nan

# ── N_λ summation (standard framework) ────────────────────────
def aho_attenuated(g, lam, coeffs):
    """Σ a_n (g/λ)^n — loop-order attenuation."""
    return sum(coeffs[n] * (g/lam)**n for n in range(len(coeffs)))

# ── Generalized weight: Σ a_n g^n w(n) ───────────────────────
def aho_gen_weighted(g, alpha_w, coeffs):
    """Σ a_n g^n × n^{-α n} — super-exponential suppression."""
    total = coeffs[0]  # n=0 term (unweighted)
    for n in range(1, len(coeffs)):
        weight = n ** (-alpha_w * n) if alpha_w > 0 else 1.0
        total += coeffs[n] * g**n * weight
    return total

# ── Comparison ────────────────────────────────────────────────
print("Anharmonic oscillator: summation methods comparison")
print(f"{'g':>6}  {'Exact':>12}  {'Trunc(10)':>12}  {'Borel-Padé':>12}  {'N_λ(best)':>12}  {'Gen(best)':>12}")
print("-" * 72)

for g in [0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0]:
    exact = aho_exact(g)
    trunc = sum(AHO_COEFFS[n] * g**n for n in range(len(AHO_COEFFS)))
    borel = aho_borel_pade(g, AHO_COEFFS)
    
    # Best N_λ
    best_nla = (1.0, float('inf'))
    for lam in np.linspace(1.0, 50, 500):
        val = aho_attenuated(g, lam, AHO_COEFFS)
        e = abs(val - exact)
        if e < best_nla[1]:
            best_nla = (lam, e)
    nla_val = aho_attenuated(g, best_nla[0], AHO_COEFFS)
    
    # Best generalized weight
    best_gen = (0.0, float('inf'))
    for alpha_w in np.linspace(0.0, 5.0, 500):
        val = aho_gen_weighted(g, alpha_w, AHO_COEFFS)
        e = abs(val - exact)
        if e < best_gen[1]:
            best_gen = (alpha_w, e)
    gen_val = aho_gen_weighted(g, best_gen[0], AHO_COEFFS)
    
    print(f"{g:>6.2f}  {exact:>12.6f}  {trunc:>12.4f}  {borel:>12.6f}  {nla_val:>12.6f}  {gen_val:>12.6f}")

print()
print("Note: with only 11 coefficients, all methods are limited.")
print("The key question is which gets closest to exact with the same info.")

## Test 3: IR Renormalon Series

The QCD pole mass has an IR renormalon ambiguity:
$\delta m \sim \sum_n n! \left(\frac{\beta_0}{2\pi}\right)^n \alpha_s^{n+1}$

This is Borel-non-summable (pole on the positive real axis).
Neither Borel nor $\mathcal{N}_\lambda$ should give a unique answer.

In [ ]:
# ── IR renormalon: Borel non-summable ─────────────────────────
# S(α) = Σ n! (β₀/(2π))^n α^{n+1}
# Borel transform: B(t) = Σ (β₀/(2π))^n t^n = 1/(1 - β₀t/(2π))
# Pole at t = 2π/β₀ on the positive real axis → Borel ambiguity

beta0 = 9.0  # nf=3
b = beta0 / (2 * np.pi)

def renormalon_truncated(alpha_s, k):
    return sum(np.math.factorial(n) * b**n * alpha_s**(n+1) for n in range(k))

def renormalon_attenuated(alpha_s, lam, k):
    return sum(np.math.factorial(n) * b**n * alpha_s**(n+1) * lam**(-(n+1)) for n in range(k))

alpha_s = 0.3
print(f"IR renormalon series at αs = {alpha_s}, β₀ = {beta0}")
print(f"Borel pole at t = 2π/β₀ = {2*np.pi/beta0:.3f}")
print()

print(f"{'k':>3}  {'Truncated':>15}  {'N_λ(λ=2)':>15}  {'N_λ(λ=5)':>15}  {'|term_k|':>15}")
print("-" * 68)
for k in range(1, 20):
    try:
        trunc = renormalon_truncated(alpha_s, k)
        att2 = renormalon_attenuated(alpha_s, 2, k)
        att5 = renormalon_attenuated(alpha_s, 5, k)
        term = np.math.factorial(k-1) * b**(k-1) * alpha_s**k
        print(f"{k:>3}  {trunc:>15.6f}  {att2:>15.6f}  {att5:>15.6f}  {term:>15.6f}")
    except OverflowError:
        print(f"{k:>3}  overflow")
        break

print()
print("For IR renormalons, both truncation and N_λ diverge.")
print("Borel summation also fails (pole on positive real axis).")
print("N_λ delays divergence but cannot resolve the fundamental")
print("ambiguity. This is expected — it's a feature, not a bug.")

## Conclusions

### Theoretical argument

For a series $S = \sum c_n g^n$ with $c_n \sim n!$, the attenuation operator
$\mathcal{N}_\lambda$ produces $\sum c_n (g/\lambda)^n$. Since $n!$ grows
super-exponentially, no fixed $\lambda < \infty$ can make this converge.
Borel summation works by dividing out the $n!$; $\mathcal{N}_\lambda$ only
rescales $g$.

### Numerical results

| Series | Borel | Optimal truncation | $\mathcal{N}_\lambda$ (optimized) | Generalized $n^{-\alpha n}$ |
|--------|-------|-------------------|----------------------------------|----------------------------|
| Euler $(-1)^n n! g^n$ | Exact | $O(e^{-1/g})$ error | Same as optimal trunc (just shifts $n_\text{opt}$) | Can beat optimal trunc |
| Anharmonic oscillator | Near-exact via Padé | Limited by $n_\text{max}$ | Coupling rescaling only | Can improve, but ad hoc |
| IR renormalon | Ambiguous (pole) | Ambiguous | Ambiguous | Ambiguous |

### Verdict

$\mathcal{N}_\lambda$ **cannot compete with Borel summation** on factorial series.
The reason is structural: Borel transforms the growth rate ($c_n \to c_n/n!$),
while $\mathcal{N}_\lambda$ only rescales the expansion parameter ($g \to g/\lambda$).
The generalized weighting $n^{-\alpha n}$ can tame factorials but breaks
the multiplicative structure that makes $\mathcal{N}_\lambda$ algebraically clean.